#MNIST DataModule

#default_exp datamodules.mnist_datamodule

This notebook defines the MNISTDataModule using fsspec for URI-based data loading.

#export

In [9]:
#| default_exp datamodules.mnist_datamodule

In [1]:
#| export
from typing import Optional, Tuple
import os

import fsspec
import torch
from pytorch_lightning import LightningDataModule
from torch.utils.data import ConcatDataset, DataLoader, Dataset, random_split
from torchvision.datasets import MNIST
from torchvision.transforms import transforms

## Data Preparation

The `prepare_data` method downloads the MNIST dataset if it's not already available. It uses `fsspec` to handle different URI schemes, caching remote data locally if necessary.

This method is called only once per GPU in distributed mode, making it ideal for downloading or syncing large datasets.

In [ ]:
#| export
from fastcore.utils import patch
from src.lightning_config import lightning_config

@lightning_config
class MNISTDataModule(LightningDataModule):
    """Example of LightningDataModule for MNIST dataset using fsspec for URI-based loading.

    A DataModule implements 5 key methods:
        - prepare_data (things to do on 1 GPU/TPU in distributed mode)
        - setup (things to do on every accelerator in distributed mode)
        - train_dataloader (the training dataloader)
        - val_dataloader (the validation dataloader(s))
        - test_dataloader (the test dataloader(s))

    This allows you to share a full dataset without explaining how to download,
    split, transform and process the data.

    Read the docs:
        https://pytorch-lightning.readthedocs.io/en/latest/extensions/datamodules.html
    """

    def __init__(
        self,
        data_uri: str = "file://./data/",
        train_val_test_split: Tuple[int, int, int] = (55_000, 5_000, 10_000),
        batch_size: int = 64,
        num_workers: int = 0,
        pin_memory: bool = False,
    ):
        super().__init__()

        # this line allows to access init params with 'self.hparams' attribute
        self.save_hyperparameters(logger=False)

        # data transformations
        self.transforms = transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        )

        self.data_train: Optional[Dataset] = None
        self.data_val: Optional[Dataset] = None
        self.data_test: Optional[Dataset] = None

    @property
    def num_classes(self) -> int:
        return 10

In [3]:
#| export
@patch
def prepare_data(self: MNISTDataModule):
    """Download data if needed, using fsspec for URI handling.

    This method is called only from a single GPU.
    Do not use it to assign state (self.x = y).
    """
    fs, path = fsspec.core.url_to_fs(self.hparams.data_uri)
    if fs.protocol != 'file':
        # For remote URIs, cache to local
        local_path = "./data"
        os.makedirs(local_path, exist_ok=True)
        fs.get(path.rstrip('/'), local_path, recursive=True)
        data_dir = local_path
    else:
        data_dir = path
    
    MNIST(data_dir, train=True, download=True)
    MNIST(data_dir, train=False, download=True)

## Data Setup

The `setup` method loads the datasets and splits them into train, validation, and test sets. It uses the URI-based data location from `prepare_data` and applies the appropriate transforms.

In [4]:
#| export
@patch
def setup(self: MNISTDataModule, stage: Optional[str] = None):
    """Load data. Set variables: `self.data_train`, `self.data_val`, `self.data_test`.

    This method is called by lightning when doing `trainer.fit()` and `trainer.test()`,
    so be careful not to execute the random split twice! The `stage` can be used to
    differentiate whether it's called before trainer.fit()` or `trainer.test()`.
    """

    # load datasets only if they're not loaded already
    if not self.data_train and not self.data_val and not self.data_test:
        fs, path = fsspec.core.url_to_fs(self.hparams.data_uri)
        if fs.protocol != 'file':
            data_dir = "./data"
        else:
            data_dir = path
        
        trainset = MNIST(
            data_dir, train=True, transform=self.transforms
        )
        testset = MNIST(
            data_dir, train=False, transform=self.transforms
        )
        dataset = ConcatDataset(datasets=[trainset, testset])
        self.data_train, self.data_val, self.data_test = random_split(
            dataset=dataset,
            lengths=self.hparams.train_val_test_split,
            generator=torch.Generator().manual_seed(42),
        )

## DataLoaders

The dataloader methods create PyTorch DataLoaders for each phase of training. The training loader shuffles the data, while validation and test loaders do not.

In [5]:
#| export
@patch
def train_dataloader(self: MNISTDataModule):
    return DataLoader(
        dataset=self.data_train,
        batch_size=self.hparams.batch_size,
        num_workers=self.hparams.num_workers,
        pin_memory=self.hparams.pin_memory,
        shuffle=True,
    )

@patch
def val_dataloader(self: MNISTDataModule):
    return DataLoader(
        dataset=self.data_val,
        batch_size=self.hparams.batch_size,
        num_workers=self.hparams.num_workers,
        pin_memory=self.hparams.pin_memory,
        shuffle=False,
    )

@patch
def test_dataloader(self: MNISTDataModule):
    return DataLoader(
        dataset=self.data_test,
        batch_size=self.hparams.batch_size,
        num_workers=self.hparams.num_workers,
        pin_memory=self.hparams.pin_memory,
        shuffle=False,
    )